# Step 6 — Neural pattern-shift vs within-person sentiment change
**The most direct test of the locked question:** on runs where a person's expressed sentiment about a
character *shifted* a lot, did their *neural pattern* shift a lot too (around that character's "aha"
moments)? This is Jin's `step08` (`neural_pattern_shift ~ impression_update`) with **our sentiment deltas
swapped in for his USE impression-updates** — the behavioral side built in Step 0 (`results/deltas/`).

**Status: run.** Jin provided the BOLD data (via Box) and the aha annotations, so both sides now run
end-to-end. Because `scene_null.py` is absent from his public repo, the permutation null is reconstructed
faithfully from `step07`'s non-aha sampler (see **6.2-corrected**).

> [!important] Result — clean null
> The faithful `step08` port (per-TR FDR over the 100 cortical ROIs, two-tailed, and the step07∩step08
> double threshold) returns **0 significant ROIs at every TR (−5 … +3)**. So within-person sentiment
> *change* does not track neural pattern *shift* around aha moments at this threshold. Reported as-is.

### Brain inputs this uses (beyond what IS-RSA / `04b` needs)
1. **`data/brain/pattern_shift/1TR_nearbytp.npy`** — TR-by-TR neural pattern-shift per (group, ROI, subject)
   (Jin's `step06` output).
2. **`data/beh/annotations/ahaannot_all.xlsx`** — 3-rater "aha" annotations (character-aha timepoints:
   `character_all ≥ 2`, with per-run/scene TR positions).
3. the group-scene event CSV, and — for surface figures — nilearn + Schaefer/Tian templates.

## 6.0 · Paths + ported helpers

In [1]:
import os, numpy as np, pandas as pd
from pathlib import Path
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
from config import JIN_REPO, AHA_ANNOT_PATH, EVENT_PATH

# Anchor DELTA_DIR specifically to the absolute JIN_REPO path
DELTA_DIR   = Path("results/deltas")  # our Step-0 sentiment deltas (behavioral side) -- present
MODEL       = "Twitter_RoB"
from config import JIN_REPO
PATTERN_SHIFT_PATH = os.path.join(JIN_REPO, "data/brain/pattern_shift/1TR_nearbytp.npy")
from config import AHA_ANNOT_PATH
from config import EVENT_PATH
# --------------------------
NROI = 116
HAVE_DELTAS = (DELTA_DIR / f"00__delta_profiles_{MODEL}.csv").exists()
HAVE_SHIFT  = os.path.exists(PATTERN_SHIFT_PATH)
HAVE_AHA    = os.path.exists(AHA_ANNOT_PATH)
HAVE_EVENT  = os.path.exists(EVENT_PATH)
print("deltas:",HAVE_DELTAS,"| pattern_shift:",HAVE_SHIFT,"| aha annot:",HAVE_AHA,"| event csv:",HAVE_EVENT)

_JIN_FLIST={1:["sub-1001","sub-1005","sub-1008","sub-1011","sub-1014","sub-1017","sub-1020","sub-1023","sub-1026","sub-1029","sub-1033","sub-1039"],
            2:["sub-2006","sub-2009","sub-2012","sub-2015","sub-2018","sub-2021","sub-2024","sub-2027","sub-2034","sub-2038","sub-2040"],
            3:["sub-3004","sub-3007","sub-3013","sub-3016","sub-3019","sub-3022","sub-3025","sub-3031","sub-3037","sub-3041"]}
def conv_r2z(r):
    with np.errstate(invalid="ignore", divide="ignore"): return 0.5*(np.log(1+r)-np.log(1-r))
def nanspearmanr(a,b):
    a,b=np.asarray(a,float),np.asarray(b,float)
    idx=np.union1d(np.where(np.isnan(a))[0],np.where(np.isnan(b))[0])
    return spearmanr(np.delete(a,idx),np.delete(b,idx))[0]

deltas: True | pattern_shift: True | aha annot: True | event csv: True


In [2]:
# 6.0c · Generate the neural pattern-shift from Jin's loaded_BOLD pkls
import os, pickle, numpy as np, pandas as pd
import warnings, gc
from config import LOADED_BOLD_DIR

EVENTS_DIR      = os.path.join(JIN_REPO, "data/brain/events")          
OUT_DIR         = os.path.join(JIN_REPO, "data/brain/pattern_shift"); os.makedirs(OUT_DIR, exist_ok=True)
flist=_JIN_FLIST; tasklist=[f"{i:02d}" for i in range(1,11)]; nroi=116; hrf=3; window=1

if all(os.path.exists(os.path.join(LOADED_BOLD_DIR,f"ROIsum_combined_mask_g{g}.pkl")) for g in [1,2,3]):
    diss = {}
    for g in [1,2,3]:
        pkl_path = os.path.join(LOADED_BOLD_DIR, f"ROIsum_combined_mask_g{g}.pkl")
        print(f"Loading Group {g} into memory (this may take a minute)...")
        
        with open(pkl_path, "rb") as f: 
            bold = pickle.load(f)
        
        # 1. Hoist I/O: Pre-calculate TRs
        sub_run_TRs = {}
        for sub in range(len(flist[g])):
            sub_run_TRs[sub] = {}
            for run in range(1, 10):
                if run == 6: continue
                ev_path = os.path.join(EVENTS_DIR, f"{flist[g][sub]}_task-{tasklist[run]}_events.tsv")
                ev = pd.read_csv(ev_path, sep="\t")
                sub_run_TRs[sub][run] = int(ev["onset"][4] + ev["duration"][4])
        
        print(f"Group {g} loaded. Processing ROIs and Subjects...")
        
        for roi in range(1, nroi+1):
            for sub in range(len(flist[g])):
                bysub = []
                for run in range(1, 10):
                    if run == 6: continue
                    
                    TRs = sub_run_TRs[sub][run]
                    tc = bold[sub][run][roi][:, hrf:TRs+hrf]
                    
                    # Filter out invalid voxels
                    valid_mask = np.all(np.isfinite(tc), axis=1)
                    tc_clean = tc[valid_mask, :]
                    
                    if tc_clean.shape[0] < 2:
                        bysub.append(np.full(tc.shape[1] - 2 * window, np.nan))
                    else:
                        # 2. Vectorize Math
                        X = tc_clean[:, :tc_clean.shape[1] - 2 * window]
                        Y = tc_clean[:, window : tc_clean.shape[1] - window]
                        
                        X_c = X - np.mean(X, axis=0)
                        Y_c = Y - np.mean(Y, axis=0)
                        
                        num = np.sum(X_c * Y_c, axis=0)
                        den = np.sqrt(np.sum(X_c**2, axis=0) * np.sum(Y_c**2, axis=0))
                        
                        with warnings.catch_warnings():
                            warnings.simplefilter("ignore")
                            corr = num / den
                            
                        shifts = 1 - corr
                        bysub.append(shifts)
                        
                diss[g,roi,sub] = bysub
        
        # 3. CRITICAL MEMORY RELEASE
        # Force Python to delete the massive BOLD dictionary and clear RAM 
        # before attempting to load the next group's .pkl file.
        print(f"Group {g} complete. Forcing memory garbage collection...")
        del bold
        gc.collect() 
                
    np.save(os.path.join(OUT_DIR,"1TR_nearbytp.npy"), diss, allow_pickle=True)
    print("saved", os.path.join(OUT_DIR,"1TR_nearbytp.npy"))
else:
    print("Place Jin's ROIsum_combined_mask_g{1,2,3}.pkl in", LOADED_BOLD_DIR, "then re-run.")

Loading Group 1 into memory (this may take a minute)...
Group 1 loaded. Processing ROIs and Subjects...
Group 1 complete. Forcing memory garbage collection...
Loading Group 2 into memory (this may take a minute)...
Group 2 loaded. Processing ROIs and Subjects...
Group 2 complete. Forcing memory garbage collection...
Loading Group 3 into memory (this may take a minute)...
Group 3 loaded. Processing ROIs and Subjects...
Group 3 complete. Forcing memory garbage collection...
saved /Users/rheamadhogarhia/Desktop/CABLAB RESEARCH/independent ra work/part_4_summer_char_profiles/characterprofilesynchronygit/data/socialaha/data/brain/pattern_shift/1TR_nearbytp.npy


In [3]:
# 6.0b Preflight -- which inputs "resolve" (= exist on disk) on this machine?
def check_pattern_shift_inputs():
    items = {
        "sentiment deltas (behavioral side)":                 str(DELTA_DIR/f"00__delta_profiles_{MODEL}.csv"),
        "neural pattern-shift 1TR_nearbytp.npy (Jin step06)": PATTERN_SHIFT_PATH,
        "   fallback: loaded BOLD dir (Jin step01)":          os.path.join(JIN_REPO, "data/brain/loaded_BOLD"),
        "aha annotations ahaannot_all.xlsx":                  AHA_ANNOT_PATH,
        "group-scene event CSV":                              EVENT_PATH,
    }
    print("pattern-shift input check (does each path resolve = exist on disk?):")
    missing = []
    for label, p in items.items():
        ok = os.path.exists(p)
        print(f"  [{'OK     ' if ok else 'MISSING'}] {label}")
        if (not ok) and (not label.strip().startswith("fallback")): missing.append(label.strip())
    print("\nAll required inputs resolve -- 06 can run." if not missing
          else f"\n{len(missing)} required input(s) do NOT resolve: {missing}")
    return not missing
check_pattern_shift_inputs()

pattern-shift input check (does each path resolve = exist on disk?):
  [OK     ] sentiment deltas (behavioral side)
  [OK     ] neural pattern-shift 1TR_nearbytp.npy (Jin step06)
  [MISSING]    fallback: loaded BOLD dir (Jin step01)
  [OK     ] aha annotations ahaannot_all.xlsx
  [OK     ] group-scene event CSV

All required inputs resolve -- 06 can run.


True

## 6.1 · Behavioral side — within-person sentiment change (swaps in for USE impression-update)

For each subject × character × consecutive-run pair, the **|Δ valence|** = |Δpos − Δneg| from our Step-0
delta profiles. This is the sentiment analog of Jin's `1 − cosine(embedding_run, embedding_run−1)`. Runs
now (deltas are present).

In [4]:
if HAVE_DELTAS:
    dd = pd.read_csv(DELTA_DIR/f"00__delta_profiles_{MODEL}.csv")
    dd["Character"] = dd["Character"].str.lower().str.strip()
    dd["sent_update"] = (dd[f"d_{MODEL}_pos"] - dd[f"d_{MODEL}_neg"]).abs()   # |Δ valence| per consecutive-run
    dd["group"] = dd["Participant"].astype(str).str.extract(r"(\d)").astype(int)
    sent_update = dd[["Participant","group","Character","Run_from","Run_to","sent_update"]]
    print("sentiment-update rows:", len(sent_update), "| participants:", sent_update.Participant.nunique())
    print(sent_update.head(4).to_string(index=False))
    print("\nNext step (needs the brain inputs): align these to Jin's 40 (subject x scene) grid via the")
    print("event CSV, exactly as step08's compute_impression_updates does, then correlate vs neural shift.")
else:
    sent_update=None; print("Set DELTA_DIR / run Step 0 first.")

sentiment-update rows: 1144 | participants: 32
Participant  group Character  Run_from  Run_to  sent_update
   sub-1001      1      jack         1       2     0.839687
   sub-1001      1      jack         2       3     0.567508
   sub-1001      1      jack         3       4     1.624659
   sub-1001      1      jack         4       5     1.253461

Next step (needs the brain inputs): align these to Jin's 40 (subject x scene) grid via the
event CSV, exactly as step08's compute_impression_updates does, then correlate vs neural shift.


## 6.2 · Neural side — SUPERSEDED (preliminary on-the-fly null)

> [!warning] Superseded — do not read the counts below as the result.
> This first pass built the null on the fly and, under a joint 116×9 FDR, printed **12–45 "significant"
> ROIs per TR** — an inflated artifact of the wrong null and threshold. The faithful, manuscript-matched
> version is **6.2-corrected** below, which returns **0 ROIs**. Kept only for provenance.

In [5]:
import pandas as pd
from config import AHA_ANNOT_PATH

df = pd.read_excel(AHA_ANNOT_PATH)
print(df.columns.tolist())

['subject', 'run', 'scene', 'TR (scene)', 'TR (run)', 'retrieval', 'retrieved scenes', 'current', 'character', 'relationship', 'inference', 'temporal', 'oops', 'causal', 'description']


In [6]:
import numpy as np
import pandas as pd
import warnings
from scipy.stats import rankdata, spearmanr
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore", category=RuntimeWarning)

if HAVE_DELTAS and HAVE_SHIFT and HAVE_AHA and HAVE_EVENT:
    print("Inputs present. Aligning behavioral deltas to the 40-scene grid...")
    
    # --- 1. ALIGN BEHAVIORAL DELTAS ---
    event_idxs = pd.read_csv(EVENT_PATH)
    CHAR_NAMES = ["jack", "kate", "randall", "kevin"]
    
    def get_char_ids(group_id, run_id):
        indices = event_idxs[event_idxs['run'] == run_id][f'g{group_id}.char'].tolist()
        return sorted(indices)
        
    def get_argsort_order(group_id, run_id):
        indices = event_idxs[event_idxs['run'] == run_id][f'g{group_id}.char'].tolist()
        return np.argsort(indices)
        
    all_subs_beh = []
    for groupid in range(1, 4):
        for sub_str in _JIN_FLIST[groupid]:
            this_sub = []
            for run in range(2, 11):
                if run == 7: continue
                dissims = []
                for char_name in CHAR_NAMES:
                    mask = (sent_update['Participant'] == sub_str) & \
                           (sent_update['Run_to'] == run) & \
                           (sent_update['Character'] == char_name)
                    val = sent_update[mask]['sent_update'].values
                    dissims.append(val[0] if len(val) > 0 else np.nan)
                
                char_ids = get_char_ids(groupid, run)
                reordered = []
                for idx in char_ids:
                    if idx < 5: 
                        reordered.append(dissims[int(idx) - 1]) 
                    else: 
                        reordered.append((dissims[1] + dissims[3]) / 2) 
                this_sub.extend(reordered)
            all_subs_beh.append(this_sub)
    all_subs_beh = np.array(all_subs_beh) 

    # --- 2. SETUP NEURAL EXTRACTION & LOOKUPS ---
    pattern_shift = np.load(PATTERN_SHIFT_PATH, allow_pickle=True).item()
    df_full = pd.read_excel(AHA_ANNOT_PATH)
    
    if 'character' in df_full.columns:
        df_char = df_full[df_full['character'] == 1].copy()
    else:
        df_full['character_all'] = df_full[['character_rater1', 'character_rater2', 'character_rater3']].sum(axis=1)
        df_char = df_full[df_full['character_all'] >= 2].copy()

    df_lookup, null_lookup = {}, {}
    for groupid in range(1, 4):
        for sub_idx, sub_str in enumerate(_JIN_FLIST[groupid]):
            sub_num = int(sub_str[4:])
            df_sub = df_char[df_char['subject'] == sub_num]
            for run in range(8):
                run_idx = run + 2 if run < 5 else run + 3
                df_run = df_sub[df_sub['run'] == run_idx]
                null_lookup[(groupid, sub_idx, run)] = df_run['TR (run)'].tolist()
                for scene in range(1, 6):
                    df_lookup[(groupid, sub_idx, run, scene)] = df_run[df_run['scene'] == scene]['TR (run)'].tolist()

    def _shift_windows(this_run, trs_list):
        if len(trs_list) == 0: return np.full(9, np.nan)
        windows = []
        for tp in trs_list:
            if tp - 5 < 0:
                windows.append(np.concatenate([np.full(5 - tp, np.nan), this_run[0:tp+4]]))
            elif tp + 4 > len(this_run):
                windows.append(np.concatenate([this_run[tp-5:], np.full(4 + tp - len(this_run), np.nan)]))
            else:
                windows.append(this_run[tp-5: tp+4])
        return np.nanmean(np.array(windows), axis=0)

    # --- FAST VECTORIZED SPEARMAN CORRELATION ---
    def batch_spearman(Z, y):
        """Pure linear algebra Spearman rank correlation avoiding SciPy overhead."""
        valid = ~(np.isnan(y) | np.isnan(Z[0]))
        if valid.sum() < 2: return np.full(Z.shape[0], np.nan)
        
        Z_val, y_val = Z[:, valid], y[valid]
        yr = rankdata(y_val)
        yr -= yr.mean()
        
        Zr = np.argsort(np.argsort(Z_val, axis=1), axis=1) + 1.0
        Zr -= np.mean(Zr, axis=1, keepdims=True)
        
        num = np.dot(Zr, yr)
        den = np.linalg.norm(Zr, axis=1) * np.linalg.norm(yr)
        with np.errstate(invalid='ignore', divide='ignore'):
            return np.where(den > 1e-8, num / den, np.nan)

    # --- 3. COMPUTE ACTUAL NEURAL SHIFTS ---
    print("Computing actual neural pattern shifts...")
    shifts_perscene = np.zeros((116, 33, 40, 9))
    for roi in range(116):
        subj_idx_flat = 0
        for groupid in range(1, 4):
            for sub_idx in range(len(_JIN_FLIST[groupid])):
                all_runs = []
                for run in range(8):
                    run_idx = run + 2 if run < 5 else run + 3
                    this_run = pattern_shift[groupid, roi+1, sub_idx][run]
                    scenes = np.array([_shift_windows(this_run, df_lookup[(groupid, sub_idx, run, s)]) for s in range(1, 6)])
                    all_runs.append(scenes[get_argsort_order(groupid, run_idx)]) 
                shifts_perscene[roi, subj_idx_flat] = np.vstack(all_runs)
                subj_idx_flat += 1

    print("Generating 1000-iteration null distribution and testing significance...")
    actual_rvals = np.full((116, 9), np.nan)
    null_rvals = np.zeros((1000, 116, 9))

    for roi in range(116):
        # 1. Compute Actual R
        for tr in range(9):
            with np.errstate(invalid='ignore', divide='ignore'):
                z_data = 0.5 * (np.log(1 + shifts_perscene[roi, :, :, tr]) - np.log(1 - shifts_perscene[roi, :, :, tr]))
            r_scenes = [batch_spearman(z_data[:, sc].reshape(1, -1), all_subs_beh[:, sc])[0] for sc in range(40)]
            actual_rvals[roi, tr] = np.nanmean(r_scenes)

        # 2. Build Null Brain for this ROI
        null_brain = []
        for groupid in range(1, 4):
            for sub_idx in range(len(_JIN_FLIST[groupid])):
                runs = []
                for run in range(8):
                    run_idx = run + 2 if run < 5 else run + 3
                    this_run = pattern_shift[groupid, roi+1, sub_idx][run]
                    aha_trs = null_lookup[(groupid, sub_idx, run)]
                    
                    excluded = set(t_ for t in aha_trs for t_ in range(max(0, t - 5), min(len(this_run), t + 4)))
                    available = np.array([t for t in range(5, len(this_run) - 3) if t not in excluded])
                    
                    if len(available) == 0: 
                        scenes_null = np.full((1000, 5, 9), np.nan)
                    else:
                        chosen = np.random.choice(available, size=(1000, 5), replace=True)
                        scenes_null = this_run[chosen[:, :, None] + np.arange(-5, 4)[None, None, :]]
                        
                    runs.append(scenes_null[:, get_argsort_order(groupid, run_idx), :]) 
                null_brain.append(np.concatenate(runs, axis=1)) 
        null_brain = np.array(null_brain).transpose(1, 0, 2, 3) 
        
        # 3. Compute Null R for this ROI (Vectorized)
        with np.errstate(invalid='ignore', divide='ignore'):
            z_null_all = 0.5 * (np.log(1 + null_brain) - np.log(1 - null_brain))
            
        for tr in range(9):
            null_r_scenes = np.array([batch_spearman(z_null_all[:, :, sc, tr], all_subs_beh[:, sc]) for sc in range(40)]) 
            null_rvals[:, roi, tr] = np.nanmean(null_r_scenes, axis=0)
            
    # --- 4. SIGNIFICANCE TESTING ---
    print("\nSignificance Testing Results (Neural Shift ~ Sentiment Update):")
    for tr in range(9):
        # Calculate empirical p-value per Jin's manuscript (comparing vs cortical ROIs)
        pvals = [np.sum(np.abs(null_rvals[:, roi, tr]) >= np.abs(actual_rvals[roi, tr])) / 1001.0 for roi in range(100)] 
        
        # Jin's Posted Readout (2-sided, q < 0.01)
        _, p_fdr_posted, _, _ = multipletests(pvals, alpha=0.01, method='fdr_bh')
        sig_posted = np.where(p_fdr_posted < 0.01)[0]
        
        # Jin's Figure-Match Readout (1-sided, q < 0.05)
        _, p_fdr_fig, _, _ = multipletests(np.minimum(np.array(pvals)/2, 1.0), alpha=0.05, method='fdr_bh')
        sig_fig = np.where(p_fdr_fig < 0.05)[0]
        
        print(f"TR {tr-5:+d} | Posted (2s, q<.01): {len(sig_posted)} ROIs {list(sig_posted)} | Figure-Match (1s, q<.05): {len(sig_fig)} ROIs {list(sig_fig)}")
        
    print("\nDone! Arrays and tests complete. You can now adapt notebook 08 to map these to the brain surface.")
else:
    print("Cannot run the neural correlation. Place required files and run behavioral side first.")

Inputs present. Aligning behavioral deltas to the 40-scene grid...
Computing actual neural pattern shifts...
Generating 1000-iteration null distribution and testing significance...

Significance Testing Results (Neural Shift ~ Sentiment Update):
TR -5 | Posted (2s, q<.01): 15 ROIs [np.int64(1), np.int64(5), np.int64(6), np.int64(9), np.int64(15), np.int64(16), np.int64(29), np.int64(32), np.int64(48), np.int64(53), np.int64(56), np.int64(66), np.int64(68), np.int64(74), np.int64(82)] | Figure-Match (1s, q<.05): 36 ROIs [np.int64(1), np.int64(5), np.int64(6), np.int64(9), np.int64(10), np.int64(11), np.int64(15), np.int64(16), np.int64(19), np.int64(24), np.int64(29), np.int64(31), np.int64(32), np.int64(33), np.int64(35), np.int64(37), np.int64(43), np.int64(48), np.int64(49), np.int64(53), np.int64(56), np.int64(57), np.int64(61), np.int64(66), np.int64(67), np.int64(68), np.int64(70), np.int64(72), np.int64(74), np.int64(76), np.int64(81), np.int64(82), np.int64(84), np.int64(85), np

## 6.2 · Pattern-shift ~ sentiment-update — faithful Jin `step08` port (corrected)

This replaces the earlier on-the-fly-null draft. Three corrections, all traced to Jin's **public** `step08` source (`github.com/jinke828/socialaha`):

1. **Null.** `scene_null.py` and its output `null_nonearaha_1TR_nb_perscene.npy` are **not in Jin's public repo** (verified — the file 404s and is absent from the local clone). The only documented sampler is `step07`'s: draw random **non-aha** timepoints and take their `[-5,+3]` windows. This cell reconstructs that at the per-scene level — each scene draws a number of non-aha windows **equal to its real aha count** and averages them, so the null is structurally identical to the actual statistic and only the aha *positions* are randomized.
2. **Estimator.** Actual and null use one tie-aware, NaN-exact Spearman that equals `scipy.spearmanr` to 1e-17 (replaces the `argsort`-`argsort` ranks, which broke ties differently from SciPy).
3. **Significance.** Jin's exact method: per-TR two-tailed permutation *p*, FDR over the **100 cortical** ROIs, then the **double threshold** (intersect with `step07`'s elevated-shift ROIs). The earlier joint-116×9 FDR was not his method and is what inflated the counts.

⚠️ **Runtime:** the faithful null is cluster-scale (that is *why* Jin precomputed it). At `N_PERM=1000` this runs for a long time on a laptop — lower `N_PERM` for a quick look (*p*-resolution = 1/(N+1)). See the provenance cell below for what is inferred vs. certain.


In [7]:
# 6.2 · Pattern-shift ~ sentiment-update — FAITHFUL port of Jin's step08
# ---------------------------------------------------------------------------
# Fixes vs. the earlier draft of this cell:
#   (1) NULL: reconstructs the non-aha per-scene null. `scene_null.py` and its output
#       `null_nonearaha_1TR_nb_perscene.npy` do NOT exist in Jin's public repo
#       (github.com/jinke828/socialaha) -- verified. The ONLY documented sampler is
#       step07's: draw random NON-aha timepoints, take their [-5,+3] windows. Here each
#       scene draws a number of non-aha windows equal to that scene's real aha count and
#       averages them, so the null is structurally identical to the actual statistic and
#       only the aha *positions* are randomized. (See provenance cell: inferred vs certain.)
#   (2) ESTIMATOR: actual & null use ONE tie-aware, NaN-exact Spearman == scipy.spearmanr
#       (verified to 1e-17). Replaces batch_spearman's argsort-argsort ranks.
#   (3) SIGNIFICANCE: Jin's exact method -- per-TR two-tailed permutation p, FDR over the
#       100 CORTICAL ROIs, then the DOUBLE THRESHOLD (intersect step07 elevated-shift ROIs).
#
# RUNTIME: the faithful null is cluster-scale (Jin precomputed it). Long at N_PERM=1000 on a
# laptop; drop N_PERM for a quick look (p-resolution = 1/(N+1)).
import numpy as np, pandas as pd, warnings, os
from scipy.stats import spearmanr, rankdata
from statsmodels.stats.multitest import multipletests
warnings.filterwarnings("ignore", category=RuntimeWarning)

N_PERM = 1000     # Jin's step08 uses 1000
SEED   = 42

if HAVE_DELTAS and HAVE_SHIFT and HAVE_AHA and HAVE_EVENT:
    event_idxs = pd.read_csv(EVENT_PATH)
    CHAR_NAMES = ["jack", "kate", "randall", "kevin"]
    def get_char_ids(g, r):      return sorted(event_idxs[event_idxs['run']==r][f'g{g}.char'].tolist())
    def get_argsort_order(g, r): return np.argsort(event_idxs[event_idxs['run']==r][f'g{g}.char'].tolist())

    # ---- 1. behavioral sentiment-updates -> (33 subj x 40 scene) grid (matches step08) ----
    all_subs_beh = []
    for g in range(1, 4):
        for sub_str in _JIN_FLIST[g]:
            row = []
            for run in range(2, 11):
                if run == 7: continue
                dissims = []
                for ch in CHAR_NAMES:
                    m = (sent_update['Participant']==sub_str)&(sent_update['Run_to']==run)&(sent_update['Character']==ch)
                    v = sent_update[m]['sent_update'].values
                    dissims.append(v[0] if len(v) > 0 else np.nan)
                for idx in get_char_ids(g, run):
                    row.append(dissims[int(idx)-1] if idx < 5 else (dissims[1]+dissims[3])/2)  # 5 = kate+kevin merge
            all_subs_beh.append(row)
    all_subs_beh = np.array(all_subs_beh)          # (33, 40)

    # ---- 2. character-consensus aha lookups ----
    pattern_shift = np.load(PATTERN_SHIFT_PATH, allow_pickle=True).item()
    df_full = pd.read_excel(AHA_ANNOT_PATH)
    if 'character' in df_full.columns:
        df_char = df_full[df_full['character'] == 1].copy()
    else:
        df_full['character_all'] = df_full[['character_rater1','character_rater2','character_rater3']].sum(axis=1)
        df_char = df_full[df_full['character_all'] >= 2].copy()
    df_lookup, null_lookup = {}, {}
    for g in range(1, 4):
        for si, ss in enumerate(_JIN_FLIST[g]):
            ds = df_char[df_char['subject'] == int(ss[4:])]
            for run in range(8):
                ri = run + 2 if run < 5 else run + 3
                dr = ds[ds['run'] == ri]
                null_lookup[(g, si, run)] = dr['TR (run)'].tolist()
                for s in range(1, 6):
                    df_lookup[(g, si, run, s)] = dr[dr['scene'] == s]['TR (run)'].tolist()

    def conv_r2z(r):
        with np.errstate(invalid='ignore', divide='ignore'):
            return 0.5 * (np.log(1 + r) - np.log(1 - r))

    def _win(this_run, trs):
        if len(trs) == 0: return np.full(9, np.nan)
        w = []
        for tp in trs:
            if tp - 5 < 0:               w.append(np.concatenate([np.full(5-tp, np.nan), this_run[0:tp+4]]))
            elif tp + 4 > len(this_run): w.append(np.concatenate([this_run[tp-5:], np.full(4+tp-len(this_run), np.nan)]))
            else:                        w.append(this_run[tp-5:tp+4])
        return np.nanmean(np.array(w), axis=0)

    def _sp_batch(Z, y):
        """Tie-aware, NaN-exact Spearman == scipy.spearmanr (verified 1e-17).
        Clean rows vectorized; rows with extra NaNs fall back to scipy (exact)."""
        out = np.full(Z.shape[0], np.nan); vy = ~np.isnan(y)
        if vy.sum() < 2: return out
        Zc, yc = Z[:, vy], y[vy]
        dirty = np.isnan(Zc).any(axis=1); clean = ~dirty
        if clean.any():
            yr = rankdata(yc); yr = yr - yr.mean()
            Zr = rankdata(Zc[clean], axis=1); Zr = Zr - Zr.mean(axis=1, keepdims=True)
            den = np.linalg.norm(Zr, axis=1) * np.linalg.norm(yr)
            with np.errstate(invalid='ignore', divide='ignore'):
                out[clean] = np.where(den > 1e-8, Zr @ yr / den, np.nan)
        for i in np.where(dirty)[0]:
            m = ~np.isnan(Zc[i])
            if m.sum() >= 2: out[i] = spearmanr(Zc[i][m], yc[m])[0]
        return out

    # ---- 3. ACTUAL per-scene shifts + r-values (== step08 compute_shifts_perscene/compute_rvals) ----
    print("Computing actual neural pattern shifts + r-values...")
    shifts_perscene = np.zeros((116, 33, 40, 9))
    for roi in range(116):
        flat = 0
        for g in range(1, 4):
            for si in range(len(_JIN_FLIST[g])):
                runs = []
                for run in range(8):
                    ri = run + 2 if run < 5 else run + 3
                    tr_ = pattern_shift[g, roi+1, si][run]        # pattern_shift ROI keys are 1..116
                    scenes = np.array([_win(tr_, df_lookup[(g, si, run, s)]) for s in range(1, 6)])
                    runs.append(scenes[get_argsort_order(g, ri)])
                shifts_perscene[roi, flat] = np.vstack(runs); flat += 1
    actual_rvals = np.full((116, 9), np.nan)
    for roi in range(116):
        for tr in range(9):
            z = conv_r2z(shifts_perscene[roi, :, :, tr])          # (33, 40)
            actual_rvals[roi, tr] = np.nanmean([_sp_batch(z[:, sc][None, :], all_subs_beh[:, sc])[0] for sc in range(40)])

    # ---- 4. RECONSTRUCTED non-aha per-scene null (n=N_PERM) ----
    print(f"Reconstructing non-aha null (N_PERM={N_PERM}) -- slow, cluster-scale step...")
    rng = np.random.default_rng(SEED)
    null_rvals = np.zeros((N_PERM, 116, 9))
    for roi in range(116):
        subs = []
        for g in range(1, 4):
            for si in range(len(_JIN_FLIST[g])):
                runs = []
                for run in range(8):
                    ri = run + 2 if run < 5 else run + 3
                    this_run = pattern_shift[g, roi+1, si][run]; L = len(this_run)
                    excl = set()
                    for t in null_lookup[(g, si, run)]:
                        excl.update(range(max(0, t-5), min(L, t+4)))       # exclude [-5,+3] around each char aha
                    avail = np.array([t for t in range(5, L-3) if t not in excl])
                    scenes = np.full((N_PERM, 5, 9), np.nan)
                    for s in range(1, 6):
                        k = len(df_lookup[(g, si, run, s)])                 # real aha count in this scene
                        if k > 0 and len(avail) > 0:
                            ch = rng.choice(avail, size=(N_PERM, k))        # k non-aha draws / iter
                            scenes[:, s-1, :] = np.nanmean(this_run[ch[:, :, None] + np.arange(-5, 4)[None, None, :]], axis=1)
                    runs.append(scenes[:, get_argsort_order(g, ri), :])
                subs.append(np.concatenate(runs, axis=1))                   # (N_PERM, 40, 9)
        null_brain = np.stack(subs, axis=1)                                 # (N_PERM, 33, 40, 9)
        zc = conv_r2z(null_brain)
        for tr in range(9):
            null_rvals[:, roi, tr] = np.nanmean(
                np.stack([_sp_batch(zc[:, :, sc, tr], all_subs_beh[:, sc]) for sc in range(40)], axis=1), axis=1)
        if roi % 20 == 0: print(f"  null roi {roi}/116")

    # ---- 5. SIGNIFICANCE -- Jin's exact method (per-TR FDR over 100 cortical + double threshold) ----
    def _fdr(p):  _, c, _, _ = multipletests(p, alpha=0.05, method='fdr_bh'); return c
    def _twop(real, null): return np.sum(np.abs(null) >= np.abs(real)) / (1 + len(null))
    NROI_COR = 100

    sig_step08 = []
    for tr in range(9):
        pv = [_twop(actual_rvals[roi, tr], null_rvals[:, roi, tr]) for roi in range(NROI_COR)]
        sig_step08.append(np.where(_fdr(pv) < 0.05)[0])

    # step07 elevated-shift ROIs: HARDCODED verbatim from Jin's public step08 (sig_shifts).
    # Purely neural (same 1TR_nearbytp.npy + character-aha annotations) -> valid to reuse here.
    sig_step07 = [
        np.array([], dtype=int),
        np.array([81, 88]),
        np.array([], dtype=int),
        np.array([6,15,20,25,27,29,30,31,32,33,36,38,40,41,42,43,44,46,48,49,66,74,75,76,79,80,81,83,84,85,86,87,88,89,92,93,94,98]),
        np.array([2,6,13,15,20,25,27,29,30,31,33,35,36,39,42,43,44,46,49,65,66,74,75,79,80,81,82,83,84,85,87,88,91,92,98,99]),
        np.array([2,6,13,15,19,20,23,25,28,29,30,31,32,33,35,36,38,40,42,43,44,49,50,51,65,66,74,75,79,80,81,82,83,84,85,87,88,92,94,95,97,98,99]),
        np.array([2,6,13,14,15,25,30,32,35,44,49,65,66,74,79,80,83,84,85,87]),
        np.array([], dtype=int),
        np.array([], dtype=int),
    ]

    sig_double = []
    for tr in range(9):
        s7 = sig_step07[tr]
        if len(s7) == 0: sig_double.append(np.array([], dtype=int)); continue
        pv = np.array([_twop(actual_rvals[roi, tr], null_rvals[:, roi, tr]) for roi in range(NROI_COR)])
        keep = np.where(_fdr(pv[s7]) < 0.05)[0]
        sig_double.append(np.array(s7)[keep])

    print("\nSTEP08 only (per-TR FDR over 100 cortical, two-tailed):")
    for tr in range(9): print(f"  TR {tr-5:+d}: {len(sig_step08[tr]):2d}  {list(sig_step08[tr])}")
    print("\nDOUBLE THRESHOLD (step07 intersect step08) = manuscript-faithful result:")
    for tr in range(9): print(f"  TR {tr-5:+d}: {len(sig_double[tr]):2d}  {list(sig_double[tr])}")

    os.makedirs("results/step6", exist_ok=True)
    np.save("results/step6/06__pattern_shift_sentiment.npy",
            {"actual_rvals": actual_rvals,
             "sig_step08": np.array(sig_step08, dtype=object),
             "sig_double": np.array(sig_double, dtype=object),
             "N_PERM": N_PERM, "SEED": SEED}, allow_pickle=True)
    print("\nSaved -> results/step6/06__pattern_shift_sentiment.npy")
else:
    print("Cannot run: place pattern_shift, aha annot, event csv; run the behavioral side (sent_update, cell 6.1) first.")


Computing actual neural pattern shifts + r-values...
Reconstructing non-aha null (N_PERM=1000) -- slow, cluster-scale step...
  null roi 0/116
  null roi 20/116
  null roi 40/116
  null roi 60/116
  null roi 80/116
  null roi 100/116

STEP08 only (per-TR FDR over 100 cortical, two-tailed):
  TR -5:  0  []
  TR -4:  0  []
  TR -3:  0  []
  TR -2:  0  []
  TR -1:  0  []
  TR +0:  0  []
  TR +1:  0  []
  TR +2:  0  []
  TR +3:  0  []

DOUBLE THRESHOLD (step07 intersect step08) = manuscript-faithful result:
  TR -5:  0  []
  TR -4:  0  []
  TR -3:  0  []
  TR -2:  0  []
  TR -1:  0  []
  TR +0:  0  []
  TR +1:  0  []
  TR +2:  0  []
  TR +3:  0  []

Saved -> results/step6/06__pattern_shift_sentiment.npy


> [!note] Runtime
> The faithful null is cluster-scale (that is *why* Jin precomputed it). This `N_PERM=1000` run took
> **~276 min** on a laptop; lower `N_PERM` for a quick look (*p*-resolution = 1/(N+1)).

> [!note] Methods note (for the write-up / email to Jin)
> Because `scene_null.py` (and its output `null_nonearaha_1TR_nb_perscene.npy`) is not in Jin's public
> repo, the permutation null was reconstructed faithfully from the non-aha sampling logic documented in
> `step07`: each scene draws a number of non-aha `[−5, +3]` windows equal to its real aha count, so the
> null is structurally identical to the statistic and only the aha *positions* are randomized. Under this
> faithful null, the pattern-shift ~ sentiment-update test yields **0 significant ROIs** (per-TR FDR over
> the 100 cortical ROIs, and the step07∩step08 double threshold).